# Week 4 Instructor Demonstration Notebook

**Purpose:** Quick demos for Segments 3, 5, 6

**Dataset:** Adult Income

**Usage:** Run cells 1-8 first to load data, then jump to specific segment demos as needed

---

## Setup: Load Adult Income Dataset

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from scipy.stats import randint, uniform
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

print("✅ All imports successful!")

In [ ]:
# Cell 2: Load Adult Income dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 
           'marital-status', 'occupation', 'relationship', 'race', 'sex',
           'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

df = pd.read_csv(url, names=columns, na_values=' ?', skipinitialspace=True)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 3 rows:")
print(df.head(3))
print(f"\nTarget distribution:")
print(df['income'].value_counts())

In [ ]:
# Cell 3: Clean data
df_clean = df.dropna()

print(f"After cleaning: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")

In [ ]:
# Cell 4: Create target and features
y = (df_clean['income'] == '>50K').astype(int)
X = df_clean.drop(['income', 'fnlwgt', 'education'], axis=1)

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")
print(f"\nClass balance: {y.value_counts(normalize=True)}")

In [ ]:
# Cell 5: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\n✅ Data split complete!")

In [ ]:
# Cell 6: Identify feature types
numeric_features = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'marital-status', 'occupation', 'relationship', 
                        'race', 'sex', 'native-country']

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"\nCategorical features ({len(categorical_features)}): {categorical_features}")

In [ ]:
# Cell 7: Create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

print("✅ Preprocessor created!")

In [ ]:
# Cell 8: Create basic pipeline (for quick demos)
basic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

print("✅ Basic pipeline ready for demos!")

---

# SEGMENT 3: Hyperparameter Tuning

**Time:** 0:45-1:15 (30 minutes)

---

## Demo 1: The Manual Tuning Problem (Don't Do This!)

**Purpose:** Show students how tedious manual tuning is

In [ ]:
# Manual tuning (tedious!)
print("Starting manual hyperparameter search...\n")

# Use numeric features only for this demo
X_train_numeric = X_train[numeric_features].copy()

best_score = 0
best_params = {}

for max_depth in [3, 5, 10, 20]:
    for min_samples_split in [2, 5, 10]:
        for n_estimators in [50, 100, 200]:
            clf = RandomForestClassifier(
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                n_estimators=n_estimators,
                random_state=42
            )
            scores = cross_val_score(clf, X_train_numeric, y_train, cv=5)
            mean_score = scores.mean()
            
            if mean_score > best_score:
                best_score = mean_score
                best_params = {
                    'max_depth': max_depth,
                    'min_samples_split': min_samples_split,
                    'n_estimators': n_estimators
                }

print(f"Best params: {best_params}")
print(f"Best score: {best_score:.3f}")
print("\n⚠️  This works, but it's verbose, error-prone, slow, and hard to extend!")

## Demo 2: GridSearchCV - The Better Way

**Purpose:** Show how GridSearchCV replaces 20 lines with 5

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Use numeric features only for this demo
X_train_numeric = X_train[numeric_features].copy()

# Step 1: Define parameter grid
param_grid = {
    'max_depth': [3, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'n_estimators': [50, 100, 200]
}

# Step 2: Create base model
base_model = RandomForestClassifier(random_state=42)

# Step 3: Create GridSearchCV
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

# Step 4: Fit (this does all the work)
print("\nFitting GridSearchCV (this will take 1-2 minutes)...\n")
grid_search.fit(X_train_numeric, y_train)

# Step 5: Get results
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.3f}")

# For test score, also use numeric features
X_test_numeric = X_test[numeric_features].copy()
print(f"Test score: {grid_search.score(X_test_numeric, y_test):.3f}")

print("\n✅ Much cleaner! And it runs in parallel with n_jobs=-1")

## Demo 3: RandomizedSearchCV for Large Spaces

**Purpose:** Show when to use RandomizedSearchCV instead of GridSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Use numeric features only for this demo
X_train_numeric = X_train[numeric_features].copy()

# Define distributions instead of lists
param_distributions = {
    'max_depth': randint(3, 20),           # Random int between 3 and 20
    'min_samples_split': randint(2, 20),
    'n_estimators': [50, 100, 200, 500],   # Can still use lists
    'max_features': uniform(0.1, 0.9)      # Random float between 0.1 and 1.0
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=20,  # Try 20 random combinations (reduced for demo speed)
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nFitting RandomizedSearchCV (faster than GridSearch)...\n")
random_search.fit(X_train_numeric, y_train)

print(f"\nBest params: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.3f}")

print("\n💡 Use RandomizedSearchCV when you have 4+ hyperparameters or continuous ranges")

## Demo 4: Interpreting cv_results_

**Purpose:** Show how to dig into all results, not just best

In [ ]:
# Convert to DataFrame for readability
results = pd.DataFrame(grid_search.cv_results_)

# Most useful columns
print("Top 10 parameter combinations:")
print(results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10))

print("\n💡 Use this to:")
print("  1. See if second-best is nearly as good (simpler model?)")
print("  2. Identify which hyperparameters matter most")
print("  3. Debug if search explored reasonable space")

## Student Hands-On Exercise

**Instructions for students:**
1. Copy the GridSearchCV code above
2. Modify param_grid to try different values
3. Run it and compare your best_params_ with the instructor's

---

# SEGMENT 5: Regularization

**Time:** 1:30-2:00 (30 minutes, after break)

---

## Demo 1: L2 Regularization (Ridge)

**Analogy:** Democracy - everyone gets a vote, but no dictator

**Effect:** Shrinks coefficients toward zero, rarely sets any to exactly zero

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Note: Using X_train, y_train from our Adult Income dataset
# For Ridge, we need to preprocess first
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Testing Ridge with different alpha values:\n")

# Try different alpha values (regularization strength)
alphas = [0.001, 0.1, 1, 10, 100]
for alpha in alphas:
    model = Ridge(alpha=alpha)
    scores = cross_val_score(model, X_train_processed, y_train, cv=5)
    print(f"Alpha={alpha:6.3f}: CV score = {scores.mean():.3f} ± {scores.std():.3f}")

print("\n💡 Alpha controls regularization strength:")
print("   - Alpha → 0: No speed limit (no regularization)")
print("   - Alpha → ∞: Severe limit (underfitting)")
print("   - Sweet spot usually between 0.1 and 10")

## Demo 2: L1 Regularization (Lasso)

**Analogy:** Hiring - you can only hire 5 people, pick the best

**Effect:** Sets many coefficients to EXACTLY zero (automatic feature selection)

In [ ]:
from sklearn.linear_model import Lasso

model = Lasso(alpha=0.01, max_iter=10000)  # Lasso needs more iterations
model.fit(X_train_processed, y_train)

# Show coefficient sparsity
print(f"Total features: {len(model.coef_)}")
print(f"Non-zero coefficients: {np.sum(model.coef_ != 0)}")
print(f"Zero coefficients: {np.sum(model.coef_ == 0)}")
print(f"Sparsity: {np.sum(model.coef_ == 0) / len(model.coef_) * 100:.1f}%")

print("\n✅ L1 automatically selected the most important features by setting others to zero!")
print("💡 Use L1 when you have many irrelevant features or need interpretability")

## Demo 3: Elastic Net (L1 + L2 Combined)

**Analogy:** Hiring with flexibility - prefer few people, but allow exceptions

**Effect:** Best of both worlds - some zeros, others shrunk

In [ ]:
from sklearn.linear_model import ElasticNet

model = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000)  # 50% L1, 50% L2
model.fit(X_train_processed, y_train)

print(f"Total features: {len(model.coef_)}")
print(f"Non-zero coefficients: {np.sum(model.coef_ != 0)}")
print(f"Sparsity: {np.sum(model.coef_ == 0) / len(model.coef_) * 100:.1f}%")

print("\n💡 l1_ratio controls the mix:")
print("   - l1_ratio=1.0: Pure L1 (Lasso)")
print("   - l1_ratio=0.0: Pure L2 (Ridge)")
print("   - l1_ratio=0.5: 50/50 blend")

## Demo 4: GridSearchCV with Regularization

**Purpose:** Show how to tune regularization type AND strength automatically

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Try L1 and L2 with different C values
# Note: In LogisticRegression, C is INVERSE of alpha (large C = weak regularization)
param_grid = {
    'penalty': ['l1', 'l2'],
    'C': [0.01, 0.1, 1, 10, 100],  # Inverse of regularization strength
    'solver': ['saga'],  # Supports both L1 and L2
    'max_iter': [1000]
}

grid = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("\nFitting GridSearchCV with regularization tuning...\n")
grid.fit(X_train_processed, y_train)

print(f"\nBest regularization: {grid.best_params_['penalty']}")
print(f"Best C: {grid.best_params_['C']}")
print(f"CV score: {grid.best_score_:.3f}")

print("\n✅ GridSearchCV found the best regularization type and strength for you!")

## Regularization Comparison Table

| Type        | Penalty       | Coefficients      | Use When...                    |
|-------------|---------------|-------------------|--------------------------------|
| L2 (Ridge)  | Squared       | Small but non-zero| Default choice, all features useful |
| L1 (Lasso)  | Absolute      | Many zeros        | Feature selection needed       |
| Elastic Net | L1 + L2       | Some zeros        | Correlated features, balanced  |

---

# SEGMENT 6: Data Leakage

**Time:** 2:00-2:25 (25 minutes)

**⚠️  MOST IMPORTANT CONCEPT OF THE DAY**

---

## Demo 1: Preprocessing Leakage - BAD Example ❌

**The #1 most common leakage pattern**

In [ ]:
# Create simple numeric-only dataset for clear demo
X_numeric = X[numeric_features].copy()
print(f"Using numeric features only for clear demo: {X_numeric.shape}\n")

# BAD: Leakage happens here!
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

print("❌ BAD APPROACH - LEAKAGE!\n")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric)  # ← Uses ALL data (train + test)!

print(f"  Scaler learned from ALL {len(X_scaled)} samples")
print(f"  Mean: {scaler.mean_[:3]}...")  # Calculated from ALL data

# Split AFTER scaling
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(X_scaled, y, random_state=42)

# Train model
model = LogisticRegression(max_iter=1000).fit(X_train_bad, y_train_bad)
print(f"\n  Test score: {model.score(X_test_bad, y_test_bad):.3f}")

print("\n⚠️  The scaler 'saw' test data! This score is FALSELY INFLATED.")
print("⚠️  Time travel analogy: We went to the future, learned test statistics, came back.")

## Demo 2: Preprocessing - CORRECT Manual Approach ✅

In [ ]:
print("✅ CORRECT APPROACH - No Leakage\n")

# Split FIRST
X_train_good, X_test_good, y_train_good, y_test_good = train_test_split(X_numeric, y, random_state=42)

# Fit scaler ONLY on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_good)  # ← Only train data

print(f"  Scaler learned from TRAIN ONLY: {len(X_train_good)} samples")
print(f"  Mean: {scaler.mean_[:3]}...")  # Different from above!

# Transform test data using TRAIN statistics
X_test_scaled = scaler.transform(X_test_good)  # ← Not fit_transform!

# Train model
model = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train_good)
print(f"\n  Test score: {model.score(X_test_scaled, y_test_good):.3f}")

print("\n✅ Test set is truly unseen. This score is HONEST.")
print("✅ No time travel!")

## Demo 3: Feature Selection Leakage - BAD Example ❌

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

print("❌ BAD: Feature selection using ALL data\n")

# BAD: Select features using ALL data
selector = SelectKBest(f_classif, k=3)
X_selected = selector.fit_transform(X_numeric, y)  # ← Uses test set labels!

print(f"  Selector used ALL {len(y)} labels to pick features")
print(f"  Selected features based on correlation with ENTIRE dataset")

# Then split
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(X_selected, y, random_state=42)

print("\n⚠️  The model knows which features correlate with test outcomes!")
print("⚠️  SEVERE leakage - would completely fail in production.")

## Demo 4: Imputation Leakage - BAD Example ❌

In [ ]:
from sklearn.impute import SimpleImputer

# Create data with missing values for demo
X_with_missing = X_numeric.copy()
X_with_missing.iloc[::50, 0] = np.nan  # Add some missing values

print(f"❌ BAD: Imputation using ALL data\n")
print(f"  Dataset has {X_with_missing.isnull().sum().sum()} missing values\n")

# BAD: Impute using ALL data's mean
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_with_missing)  # ← Test set influences mean

print(f"  Imputer used mean from ALL data: {imputer.statistics_[:3]}...")

X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(X_imputed, y, random_state=42)

print("\n⚠️  The mean used for imputation includes test set values. Leakage!")

## Demo 5: Pipeline - The Leakage-Proof Solution ✅

**Pipeline makes leakage IMPOSSIBLE by enforcing correct order automatically**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("✅ CORRECT: Using Pipeline to prevent leakage\n")

# Split FIRST (Pipeline doesn't change this)
X_train_pipe, X_test_pipe, y_train_pipe, y_test_pipe = train_test_split(X_numeric, y, random_state=42)

# Define Pipeline (works on raw Adult Income data)
pipe = Pipeline([
    ('scaler', StandardScaler()),      # Step 1: Scaling
    ('classifier', LogisticRegression(max_iter=1000))  # Step 2: Model
])

print("Pipeline structure:")
print(pipe)

# Fit Pipeline (internally does scaler.fit_transform on X_train, then trains model)
print("\nFitting pipeline...")
pipe.fit(X_train_pipe, y_train_pipe)

print("  → Pipeline called scaler.fit_transform(X_train)")
print("  → Scaler ONLY saw training data!")
print("  → Then trained classifier on scaled training data")

# Predict (internally does scaler.transform on X_test, then predicts)
score = pipe.score(X_test_pipe, y_test_pipe)

print(f"\nTest score: {score:.3f}")
print("  → Pipeline called scaler.transform(X_test) (not fit_transform!)")
print("  → Used TRAIN statistics to scale test data")
print("  → Then made predictions")

print("\n✅ Impossible to mess up! Pipeline handles it automatically.")

## Demo 6: Pipeline + ColumnTransformer (Preview)

**Handle different preprocessing for different columns - still no leakage!**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Use full dataset with mixed types
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X, y, random_state=42)

# Define transformers for different column types
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])

# Put in Pipeline
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

print("Advanced Pipeline with ColumnTransformer:")
print(pipe)

# Use as before - no leakage!
print("\nFitting pipeline with mixed data types...")
pipe.fit(X_train_full, y_train_full)

score = pipe.score(X_test_full, y_test_full)
print(f"\nTest score: {score:.3f}")

print("\n✅ Handles numeric (scaling) and categorical (one-hot) separately")
print("✅ Still no leakage - all fitting done on training data only")
print("\n💡 You'll build this fully in Segment 7!")

## Student Hands-On Exercise

**Instructions for students:**
1. Create a Pipeline with:
   - StandardScaler
   - Any classifier (LogisticRegression, DecisionTree, RandomForest)
2. Fit on training data
3. Evaluate on test data
4. Verify it runs without errors

## Data Leakage Prevention Checklist

**Print this and keep it visible!**

- [ ] Split data FIRST (train/test or CV folds)
- [ ] Fit all preprocessing ONLY on training data
- [ ] Use Pipeline to enforce correct order
- [ ] Never use fit_transform on test set (only transform)
- [ ] Don't include target-derived features
- [ ] Don't shuffle time series data randomly
- [ ] Verify test accuracy ≤ train accuracy (usually)
- [ ] Test in production-like conditions

**If any box unchecked → LEAKAGE RISK!**

---

## End of Instructor Demos

**Next Steps:**
- **Segment 7:** Build complete production pipeline from scratch together
- **Segment 8:** Students practice with pair programming
- **Homework:** Apply all concepts to new dataset (Titanic)

---